# Autoencoder Training With PyEarthTools

This tutorial extends the autoencoder workflow by using PyEarthTools for both data loading and model training. The data pipeline prepares Himawari satellite imagery, `PipelineLightningDataModule` turns that pipeline into PyTorch dataloaders, and `pyearthtools.training.lightning.Train` runs a PyTorch Lightning training loop.

The model and `LightningWrapper` are defined in this notebook, so this tutorial is self contained and does not require code from any other project.

In [ ]:
import datetime
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import lightning as L

import pyearthtools.data as petdata
import pyearthtools.pipeline as petpipe
import pyearthtools.training

In [ ]:
warnings.filterwarnings("ignore")
torch.manual_seed(42)

device = "gpu" if torch.cuda.is_available() else "cpu"
accelerator = "gpu" if torch.cuda.is_available() else "cpu"
workdir = Path("AutoEncoder_PET_Training_Output")
workdir.mkdir(exist_ok=True)

## Site Archive

Select the site archive module for the system where the notebook is running. The original tutorial uses JASMIN; at NCI, change this import to the NCI site archive.

In [ ]:
import site_archive_jasmin

## Build The Data Pipeline

The pipeline reads Himawari surface global irradiance, sorts and aligns dimensions, selects a smaller spatial domain, normalises values, converts the result to NumPy, and reshapes the sample to the PyTorch image convention: time, channel, height, width.

In [ ]:
himawari = petdata.archive.Himawari("surface_global_irradiance")

training_pipeline = petpipe.Pipeline(
    himawari,
    petpipe.operations.xarray.Sort(order=["time", "latitude", "longitude"]),
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),
    petdata.transform.region.Bounding(-35, -25, 138, 150),
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    petpipe.operations.xarray.conversion.ToNumpy(),
    petpipe.operations.numpy.reshape.Rearrange("c t h w -> t c h w"),
    iterator=petpipe.iterators.DateRange("20200101T00", "20200301T00", interval="10 minutes"),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

training_pipeline

In [ ]:
selected_date = datetime.datetime(2021, 6, 9, 2, 0)
eda_pipeline = petpipe.Pipeline(
    himawari,
    petpipe.operations.xarray.Sort(order=["time", "latitude", "longitude"]),
    petpipe.operations.xarray.AlignDataVariableDimensionsToDatasetCoords(),
    petdata.transform.region.Bounding(-35, -25, 138, 150),
    petpipe.operations.xarray.normalisation.SingleValueDivision(1200),
    iterator=petpipe.iterators.DateRange("20210101T00", "20210103T00", interval="1 hour"),
    exceptions_to_ignore=petdata.exceptions.DataNotFoundError,
)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), constrained_layout=True)
for i, ax in enumerate(axes):
    plot_time = selected_date + datetime.timedelta(hours=i)
    eda_pipeline[plot_time]["surface_global_irradiance"][0].plot.imshow(ax=ax)
    ax.set_title(str(plot_time))

## Define Train And Validation Splits

The pipeline itself is reusable; the PET Lightning data module below swaps in the training or validation iterator as needed.

In [ ]:
train_split = petpipe.iterators.DateRange("20200101T00", "20200215T00", interval="10 minutes")
valid_split = petpipe.iterators.DateRange("20200215T00", "20200301T00", interval="10 minutes")

batch_size = 8
num_workers = 0
max_epochs = 3

## Define The Autoencoder

This compact convolutional autoencoder reconstructs the input image. The final interpolation makes the output spatial shape match the input even when the input dimensions are odd.

In [ ]:
class AutoEncoder(nn.Module):
    def __init__(self, input_channel_count=1, output_channel_count=1):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(input_channel_count, 16, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=7, padding=3),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(32, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False),
            nn.Conv2d(16, output_channel_count, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        input_size = x.shape[-2:]
        latent = self.encoder(x)
        reconstructed = self.decoder(latent)
        return F.interpolate(reconstructed, size=input_size, mode="bilinear", align_corners=False)

## Wrap The Model For Lightning

PET's training wrapper expects a Lightning module. This `LightningWrapper` keeps the autoencoder architecture separate from the training, validation, prediction, loss, and optimiser logic.

In [ ]:
class LightningWrapper(L.LightningModule):
    def __init__(self, model, lr=1e-3):
        super().__init__()
        self.model = model
        self.lr = lr
        self.criterion = nn.L1Loss()

    def _unpack_batch(self, batch):
        x = batch
        while isinstance(x, (tuple, list)):
            x = x[0]
        if isinstance(x, dict):
            x = next(iter(x.values()))
        x = torch.as_tensor(x).float()
        if x.ndim == 5 and x.shape[1] == 1:
            x = x[:, 0]
        return torch.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch, stage):
        x = self._unpack_batch(batch)
        x_hat = self(x)
        loss = self.criterion(x_hat, x)
        self.log(f"{stage}_loss", loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, "train")

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, "valid")

    def predict_step(self, batch, batch_idx):
        x = self._unpack_batch(batch)
        x_hat = self(x)
        return {"x": x.detach().cpu(), "x_hat": x_hat.detach().cpu()}

    def configure_optimizers(self):
        return optim.Adam(self.parameters(), lr=self.lr)

In [ ]:
base_model = AutoEncoder(input_channel_count=1, output_channel_count=1)
model = LightningWrapper(base_model, lr=1e-3)
model

## Create The PET Lightning Data Module

`PipelineLightningDataModule` converts the PET pipeline and date iterators into Lightning-compatible dataloaders. The model sees batches of tensors produced from the same pipeline used for exploration.

In [ ]:
data_module = pyearthtools.training.data.lightning.PipelineLightningDataModule(
    training_pipeline,
    train_split=train_split,
    valid_split=valid_split,
    batch_size=batch_size,
    num_workers=num_workers,
)

data_module

## Train With PET

`pyearthtools.training.lightning.Train` owns the Lightning trainer setup, checkpoint/log directory, and dataloader connection. Setting `load=False` starts a fresh run rather than resuming from an existing checkpoint in `workdir`.

In [ ]:
trainer = pyearthtools.training.lightning.Train(
    model,
    data_module,
    path=workdir,
    trainer_kwargs={
        "max_epochs": max_epochs,
        "accelerator": accelerator,
        "devices": 1,
        "num_sanity_val_steps": 0,
        "logger": False,
        "enable_checkpointing": False,
        "enable_model_summary": False,
    },
)

trainer.fit(load=False)

## Predict And Plot Reconstructions

After training, use the Lightning trainer directly for a small validation batch and compare the input, reconstruction, and reconstruction error.

In [ ]:
validation_loader = data_module.val_dataloader()
predictions = trainer.trainer.predict(model, dataloaders=validation_loader)

x = torch.cat([batch["x"] for batch in predictions], dim=0).numpy()
x_hat = torch.cat([batch["x_hat"] for batch in predictions], dim=0).numpy()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(12, 10), constrained_layout=True)
for row in range(3):
    axes[row, 0].imshow(x[row, 0], vmin=0, vmax=1)
    axes[row, 0].set_title("Input")
    axes[row, 1].imshow(x_hat[row, 0], vmin=0, vmax=1)
    axes[row, 1].set_title("Reconstruction")
    axes[row, 2].imshow(x_hat[row, 0] - x[row, 0], cmap="RdBu_r")
    axes[row, 2].set_title("Error")
    for ax in axes[row]:
        ax.set_axis_off()

## Next Steps

The same pattern can be reused for more realistic models: keep the PET pipeline responsible for data preparation, keep the Lightning module responsible for model-specific training logic, and let `pyearthtools.training.lightning.Train` run the training loop.